# Modèle de langage et génération de séquence

Dans ces travaux pratiques, nous allons étudier les différentes manières de décoder du texte à partir d'un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage : traduction, question/réponse, génération…

Nous prendrons l'exemple de la génération de texte avec GPT-2, un transformer de type *décodeur seul* : les grands modèles de langage actuels (GPT-4, Llama, Mistral…) reposent sur le même principe et se décodent avec les mêmes méthodes.

Nous partirons de la méthode la plus simple pour progresser jusqu'aux méthodes utilisées en pratique, en codant nous-mêmes le choix du token suivant. Cela vous permettra de bien comprendre le rôle de la softmax en sortie des modèles de langage. Pour finir, nous retrouverons tous ces réglages dans la méthode `generate` de la bibliothèque `transformers`.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de Hugging Face peut servir de ressource complémentaire à ce notebook.

Ce TP n'a pas besoin de carte graphique : GPT-2 small tourne sur CPU, et chaque test sur les 7 prompts du TP y prend moins d'une minute. Il utilise le GPU s'il y en a un.

**Attention** : ces modèles ont été entraînés sur des textes du web, sans filtrage. Les textes générés, surtout par échantillonnage, peuvent être incohérents, faux ou choquants.

## Installation

La bibliothèque [`transformers`](https://github.com/huggingface/transformers) de [Hugging Face](https://huggingface.co/) met à disposition un grand nombre de modèles de la famille des transformers. Elle est préinstallée sur Colab ; la cellule suivante l'installe ailleurs (Codespace, poste personnel).

In [ ]:
!pip install -q transformers

## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles pré-entraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste du TP.

In [ ]:
lang = "fr"
# lang = "en"

## Import d'un modèle entraîné

En plus de mettre à disposition le code source de nombreuses architectures de transformers, la bibliothèque `transformers` permet de récupérer des [modèles entraînés](https://huggingface.co/models) depuis le Hub Hugging Face. La méthode `from_pretrained` d'`AutoModelForCausalLM` récupère l'architecture et les poids d'un modèle qui prédit le token suivant (*causal language model*), celle d'`AutoTokenizer` son tokeniseur.

- En anglais, `"gpt2"` est le GPT-2 small d'OpenAI (2019).
- En français, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) est un GPT-2 small entraîné sur un corpus français.

Les modèles PyTorch sont des `nn.Module` : on les déplace sur le GPU avec `.to(device)`, et on les passe en mode évaluation avec `.eval()` (le dropout est alors désactivé).

Les textes que voit un GPT-2 pendant son entraînement sont séparés par un symbole spécial de début et de fin de texte : `<|endoftext|>` pour GPT-2, `</s>` pour BelGPT-2. La cellule garde son index dans `start_token_id`.

In [ ]:
import functools
import statistics

import torch
import tqdm.auto
import transformers

# Mêmes tirages aléatoires à chaque exécution du notebook
torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")

pretraining_name = "antoinelouis/belgpt2" if lang == "fr" else "gpt2"

model = transformers.AutoModelForCausalLM.from_pretrained(pretraining_name)
model = model.to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(pretraining_name)

# Symbole de début et de fin de texte
start_token_id = tokenizer.convert_tokens_to_ids(
  "</s>" if lang == "fr" else "<|endoftext|>"
)

## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques. Pour cela, on peut afficher sa configuration et le modèle lui-même (un `nn.Module` s'affiche avec toutes ses couches).

- *Combien le modèle a-t-il de couches de transformer ?*
- *Quels sont ses mécanismes de régularisation ?*
- *De stabilisation de l'apprentissage ?*
- *Combien de paramètres possède-t-il ?*
- *La configuration donne l'index du symbole de début de texte dans `bos_token_id`. Est-ce bien `start_token_id` ? Affichez le token qui correspond à chacun avec `tokenizer.convert_ids_to_tokens`, en anglais puis en français.*

In [ ]:
print(model.config)
print(model)

*Votre réponse à compléter ici.*
-
-
-
-
-

### Solution

In [ ]:
n_parameters = sum(p.numel() for p in model.parameters())
print(f"{n_parameters / 1e6:.0f} millions de paramètres")
print(f"Couches : {model.config.n_layer}")
for name, index in [
  ("bos_token_id", model.config.bos_token_id),
  ("start_token_id", start_token_id),
]:
  print(f"{name} : {index} = {tokenizer.convert_ids_to_tokens(index)!r}")

- Il possède 12 couches de transformer (`n_layer`, ou les 12 `GPT2Block` affichés).
- Son principal mécanisme de régularisation est le dropout (`attn_pdrop`, `resid_pdrop`, `embd_pdrop`).
- Ses mécanismes de stabilisation de l'apprentissage sont la normalisation de couche (`LayerNorm`, `ln_1`, `ln_2`) et les connexions résiduelles.
- Il a environ 124 millions de paramètres. La couche de sortie `lm_head` n'en ajoute aucun : elle réutilise la matrice d'embedding des tokens de l'entrée (`wte`), et `parameters()` ne la compte qu'une fois.
- Pour GPT-2, `bos_token_id` vaut 50256, `<|endoftext|>` : c'est bien `start_token_id`. Pour BelGPT-2, `bos_token_id` vaut aussi 50256, mais ce token est `ĠArcade` dans son vocabulaire : sa configuration a gardé la valeur de GPT-2, alors que son symbole de début et de fin de texte est `</s>`, d'index 2. Un modèle pré-entraîné n'est pas toujours cohérent avec son tokeniseur : vérifiez ses symboles spéciaux avant de vous en servir.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on procède de manière itérative : on génère les tokens un par un, en commençant par un texte vide ou par une phrase que l'on souhaite compléter.

Pour générer un token, on donne en entrée du réseau les tokens générés jusque-là (sous forme d'indices dans le vocabulaire). Le réseau produit alors un score par token du vocabulaire. On choisit l'un de ces tokens, on l'ajoute à ce qui a déjà été généré, puis on recommence pour générer le token suivant.

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

Tout l'enjeu du décodage est de trouver le texte qui maximise ce produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approcher.

C'est le **masque causal** du transformer qui rend cette décomposition possible : pendant l'entraînement, chaque position ne voit que les positions précédentes.

## Tokenisation

Le tokeniseur récupéré avec le modèle transforme une phrase en indices de vocabulaire (des *sous-mots*) que le modèle comprend. Dans l'affichage des tokens, `Ġ` marque un token qui commence par une espace.

Nous utiliserons dans la suite du TP les fonctions `encode` et `decode` suivantes. `encode` ajoute le symbole de début de texte, et renvoie un tenseur de forme `(1, nombre de tokens)` sur le bon device.

BelGPT-2 a été entraîné sur des textes où la ponctuation est séparée des mots par des espaces (« j' ai », « fin . ») : ses générations en gardent la trace.

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  """Indices des tokens, après le symbole de début : forme (1, nombre de tokens)."""
  tokens = tokenizer.encode(sentence, add_special_tokens=False, return_tensors="pt")
  start = torch.tensor([[start_token_id]])
  return torch.cat([start, tokens], dim=1).to(device)


def decode(tokens: torch.Tensor) -> str:
  """Texte des tokens, sans le symbole de début."""
  return tokenizer.decode(tokens.squeeze(0)[1:])


example = encode(
  "Je pense, donc je suis" if lang == "fr" else "I think, therefore I am"
)
print(example)
print(tokenizer.convert_ids_to_tokens(example[0]))

## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain token. Étudions son comportement.

Comme tout module PyTorch, on calcule la passe forward avec `model(token_ids)`. Le résultat est un objet dont les attributs principaux sont :

- `logits` : les scores du modèle, avant softmax ;
- `past_key_values` : un cache des clés et valeurs d'attention déjà calculées.

*Appliquez le modèle à diverses séquences, par exemple :*

- *« » (le texte vide)*
- *« Je pense, donc je suis » ou « I think, therefore I am » en fonction de la langue du modèle*

*Quelle est la forme des `logits` ? À quoi correspond chaque dimension ?*

*Désactivez le calcul des gradients : nous n'entraînons pas le modèle.*

**Documentation utile**

- [`GPT2LMHeadModel`](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)
- [`torch.no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `encode(sentence)` donne les `token_ids`, de forme `(1, nombre de tokens)`
- `with torch.no_grad():` : les calculs du bloc ne gardent pas de quoi calculer les gradients
- `output = model(token_ids)`, puis `output.logits.shape` donne la forme des logits
- Résultat attendu pour le texte vide : `(1, 1, 50257)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def describe_shapes(sentence: str) -> None:
  with torch.no_grad():
    output = model(...)
  print(f"Forme des logits pour '{sentence}' : {tuple(...)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")
```

### Solution

In [ ]:
def describe_shapes(sentence: str) -> None:
  with torch.no_grad():
    output = model(encode(sentence))
  print(f"Forme des logits pour '{sentence}' : {tuple(output.logits.shape)}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

Les logits ont la forme `(batch, nombre de tokens, taille du vocabulaire)` : pour chaque position de l'entrée, le modèle donne un score à chacun des 50 257 tokens du vocabulaire pour la position **suivante**. Le texte vide a une position, celle du symbole de début qu'`encode` ajoute. Pour générer, seule la dernière position nous intéresse.

## Fonction `forward` adaptée aux besoins du décodage

Nous utiliserons la fonction `forward` suivante, qui rend un résultat adapté au décodage : seule la dernière position nous intéresse (la prédiction du token suivant).

Elle utilise le cache `past_key_values` : il évite de recalculer, à chaque nouveau token, les clés et valeurs d'attention de tous les tokens précédents. Quand le cache est donné, il ne faut passer au modèle **que les nouveaux tokens**.

In [ ]:
@torch.no_grad()
def forward(tokens: torch.Tensor, past=None) -> tuple[torch.Tensor, object]:
  """Logits du token suivant, de forme (1, taille du vocabulaire), et le cache."""
  output = model(tokens, past_key_values=past, use_cache=True)
  return output.logits[:, -1, :], output.past_key_values

Il faut donc utiliser au choix :

- `forward(all_tokens)`
- `forward(last_token, past)`

où `all_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7)` là où `last_token` serait de forme `(1, 1)`. Les logits renvoyés ont la forme `(1, taille du vocabulaire)` : la dimension du batch, de taille 1, est conservée.

*Pourquoi est-il souvent utile de conserver des dimensions à `1` dans les réseaux de neurones ?*

*Votre réponse à compléter ici.*

### Solution

Ces dimensions permettent de calculer en parallèle des résultats pour plusieurs exemples (batching) : le même code fonctionne pour un batch de 1 comme de 64.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Passe forward sur ce qui a été produit jusqu'ici (entrées et sorties précédentes ; grâce au cache, seulement le dernier token après le premier pas)
2. Choix de l'indice du token suivant
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (nous utiliserons seulement le nombre de tokens produits dans ce TP)

L'étape 2 est celle où tout se joue. Le code suivant implémente tout le reste, afin de pouvoir nous concentrer sur l'étape 2 : chaque méthode de décodage sera une fonction qui reçoit les logits du dernier token (forme `(1, taille du vocabulaire)`) et renvoie l'indice du token choisi (un tenseur de forme `()`). Décorée par `@decoding_loop`, elle devient une fonction `(prompt, nombre de tokens à générer, paramètres de la méthode)` qui renvoie le texte généré.

In [ ]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    decoded = [encode(prompt)]
    past = None
    for _ in tqdm.auto.trange(length, desc="Tokens", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded.append(index.reshape(1, 1))
    return decode(torch.cat(decoded, dim=1))

  return wrapper

## Décodage glouton (*greedy*)

Le décodage glouton est la méthode la plus simple pour décoder du texte depuis un modèle appris : à chaque étape, on prend le token de plus forte probabilité.

Dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

le décodage glouton choisit $\text{Token}_i$ pour maximiser $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un token moins probable pour ensuite atteindre des tokens très probables.

L'exemple suivant est tiré de l'article de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

Après « The », on choisit « nice » parce que c'est le mot le plus probable, alors que le texte complet « The nice woman » a une probabilité (0,20) inférieure à celle qu'on aurait obtenue en choisissant « dog » (« The dog has », 0,36).

*Codez la fonction `greedy(logits)` qui prend en entrée les logits du modèle et renvoie l'indice du token choisi, un tenseur de forme `()`. Elle utilisera `torch.argmax`. Générez 10 tokens après « Je suis allé au » (« I went to the » en anglais).*

**Documentation utile**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.argmax(logits, dim=-1)` : l'indice du plus grand logit de chaque ligne, de forme `(1,)` ; `[0]` en extrait le tenseur de forme `()`
- Inutile de passer par la softmax : elle ne change pas l'ordre des scores
- `greedy(prompt, 10)` : la fonction décorée prend le prompt et le nombre de tokens à générer

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return ...


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))
```

### Solution

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return torch.argmax(logits, dim=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

## Test sur des prompts variés

La fonction `test_decoding` suivante applique une méthode de décodage, avec ses paramètres, à chacun des prompts de la liste `inputs`. Elle affiche les générations et les renvoie dans une liste ; `show` les affiche.

*Exécutez les cellules suivantes, qui testent le décodage glouton avec 50 tokens par prompt. Que constatez-vous ?*

In [ ]:
if lang == "fr":
  inputs = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  inputs = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [ ]:
def show(generations: list[str]) -> None:
  """Affiche chaque prompt de `inputs` avec sa génération."""
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)


def test_decoding(function, *args, **kwargs) -> list[str]:
  """Applique la méthode de décodage `function` à chaque prompt de `inputs`."""
  generations = [
    function(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)
  ]
  show(generations)
  return generations

In [ ]:
greedy_generations = test_decoding(greedy, 50)

## Mesurer la répétition

Les générations gloutonnes « bouclent » vite : elles répètent les mêmes groupes de mots. Pour comparer les méthodes de décodage sur ce point, mesurons-le par la **part de 3-grammes distincts** d'un texte : on découpe le texte en mots, on forme tous les triplets de mots consécutifs (les 3-grammes), et on divise le nombre de triplets différents par le nombre total de triplets. Elle vaut 1 pour un texte sans répétition, et baisse quand des passages se répètent.

*Codez la fonction `distinct_3grams(text)` qui renvoie la part de 3-grammes distincts de `text`. Affichez ensuite sa moyenne sur les générations gloutonnes, `greedy_generations`, avec `statistics.mean`. Les sections suivantes la réutilisent pour chaque méthode.*

In [ ]:
def distinct_3grams(text: str) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `text.split()` : la liste des mots, séparés par les espaces
- `zip(words, words[1:], words[2:])` : les triplets de mots consécutifs
- `set(...)` : les triplets différents
- `statistics.mean(map(distinct_3grams, greedy_generations))` : la moyenne sur les générations
- Exemple à vérifier : `"a b c a b c"` a 4 triplets, dont 3 différents : 0,75

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(...)
  return ... / ...


print(distinct_3grams("a b c a b c"))
print(f"Glouton : {statistics.mean(...):.2f}")
```

### Solution

In [ ]:
def distinct_3grams(text: str) -> float:
  words = text.split()
  trigrams = list(zip(words, words[1:], words[2:]))
  return len(set(trigrams)) / len(trigrams)


print(distinct_3grams("a b c a b c"))
print(f"Glouton : {statistics.mean(map(distinct_3grams, greedy_generations)):.2f}")

Plusieurs générations gloutonnes répètent la même phrase, ou le même groupe de mots, jusqu'à la fin : c'est une faiblesse très classique du décodage glouton. La part de 3-grammes distincts le chiffre : elle est nettement sous 1, là où un texte écrit par un humain en est proche.

## Échantillonnage (*sampling*)

Les résultats du décodage glouton bouclent rapidement et sont souvent très génériques. Les méthodes suivantes cherchent à corriger ces défauts.

La première consiste à **échantillonner** à partir des probabilités du modèle, plutôt que de toujours choisir le token le plus probable : un token de probabilité 0,3 est tiré 3 fois sur 10.

*Reprenez la forme de la fonction `greedy`, mais au lieu de choisir l'élément maximal, calculez une distribution de probabilités avec `torch.softmax`, puis tirez le prochain token selon cette distribution avec `torch.multinomial`, dans une fonction `sampling(logits)`. Testez-la avec `test_decoding` (50 tokens par prompt), et comparez sa part de 3-grammes distincts à celle du décodage glouton.*

**Documentation utile**

- [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)
- [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.softmax(logits, dim=-1)` : les probabilités, de forme `(1, taille du vocabulaire)`
- `torch.multinomial(probabilities, num_samples=1)` : un tirage par ligne, de forme `(1, 1)` ; `[0, 0]` en extrait le tenseur de forme `()`
- `statistics.mean(map(distinct_3grams, generations))` : la part de 3-grammes distincts

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = ...
  return torch.multinomial(...)[0, 0]


generations = test_decoding(sampling, 50)
print(f"3-grammes distincts : {...:.2f}")
```

### Solution

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = torch.softmax(logits, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


generations = test_decoding(sampling, 50)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

Les générations ne bouclent plus : la part de 3-grammes distincts est proche de 1. En contrepartie, elles perdent souvent le fil, parce qu'un token peu probable est tiré de temps en temps et que la suite du texte doit s'en accommoder.

## Température

Avant la softmax, on peut diviser les logits par une **température** $T$ :

- $T < 1$ accentue les écarts : le modèle choisit plus souvent les tokens les plus probables (plus cohérent, moins varié) ;
- $T > 1$ les aplatit : les tokens rares sont plus souvent tirés (plus varié, moins cohérent) ;
- $T \to 0$ revient au décodage glouton.

*Ajoutez un paramètre `temperature` à votre fonction d'échantillonnage, dans une nouvelle fonction `temperature_sampling(logits, temperature)`. Comparez les générations et leur part de 3-grammes distincts pour $T = 0.5$ et $T = 1.5$.*

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `logits / temperature` : les logits divisés par la température, avant la softmax
- `temperature_sampling(prompt, 50, temperature=0.5)` : la fonction décorée transmet ses paramètres nommés à la vôtre ; `test_decoding` aussi

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(..., dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 50, temperature=temperature)
  print(f"T = {temperature} : {...:.2f} de 3-grammes distincts")
```

### Solution

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(logits / temperature, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = test_decoding(temperature_sampling, 50, temperature=temperature)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"T = {temperature} : {repetition:.2f} de 3-grammes distincts")

À $T = 0.5$, les générations sont plus cohérentes, sans boucler encore sur 50 tokens ; plus $T$ baisse, plus elles se rapprochent du décodage glouton et de ses boucles. À $T = 1.5$, elles n'ont presque plus de sens : des tokens rares sont tirés à chaque pas. La température règle ce compromis entre cohérence et variété.

## Top-k sampling

Une variation de l'échantillonnage consiste à ne tirer que parmi les `k` tokens les plus probables, pour éviter de générer un token très improbable.

*Repartez de la fonction `sampling` pour coder la fonction `k_sampling(logits, k)` qui implémente cette amélioration avec `torch.topk`. Testez-la avec `test_decoding`, pour `k = 20`.*

**Documentation utile**

- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `values, indices = torch.topk(logits, k=k, dim=-1)` : les `k` plus grands logits, par ordre décroissant, et leurs indices dans le vocabulaire, tous deux de forme `(1, k)`
- `torch.softmax(values, dim=-1)` : les probabilités des `k` tokens gardés
- `torch.multinomial(probabilities, num_samples=1)[0, 0]` : la position tirée parmi les `k`
- `indices[0, position]` : l'indice dans le vocabulaire du token tiré

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(...)
  probabilities = torch.softmax(..., dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[...]


generations = test_decoding(k_sampling, 50, k=20)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values, dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[0, sampled]


generations = test_decoding(k_sampling, 50, k=20)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Top-p sampling

Une autre amélioration consiste à ne garder que les tokens les plus probables dont la probabilité cumulée atteint `p` (le *noyau* de la distribution), et pas les suivants. Le nombre de tokens candidats s'adapte alors à la confiance du modèle : peu de candidats quand un token domine, beaucoup quand la distribution est plate.

*Repartez de la fonction `k_sampling` et ajoutez-y le top-p sampling, dans la fonction `k_p_sampling(logits, k, p)`. Vous pourrez utiliser `torch.cumsum`. Testez-la avec `test_decoding`, pour `k = 20` et `p = 0.85`.*

**Documentation utile**

- [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html)

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.softmax(values[0], dim=-1)` : les probabilités des `k` tokens gardés, par ordre décroissant, de forme `(k,)`
- `torch.cumsum(probabilities, dim=0) < p` : vrai pour les tokens avant le seuil ; `.sum()` les compte, et on ajoute 1 pour garder le token qui franchit `p`
- `int(...)` convertit ce nombre en entier Python, pour s'en servir dans une tranche
- Exemple à vérifier : pour les probabilités `[0.5, 0.3, 0.15, 0.05]` et `p = 0.85`, les sommes cumulées sont `[0.5, 0.8, 0.95, 1.0]` : 2 sont sous `p`, on garde 3 tokens
- `torch.multinomial(probabilities[:n_kept], num_samples=1)[0]` tire parmi les `n_kept` premiers ; `torch.multinomial` n'a pas besoin de probabilités qui somment à 1

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = ...
  n_kept = int((...).sum()) + 1
  sampled = torch.multinomial(..., num_samples=1)[0]
  return indices[0, sampled]


generations = test_decoding(k_p_sampling, 50, k=20, p=0.85)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")
```

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values[0], dim=-1)
  # Nombre de tokens nécessaires pour que la probabilité cumulée atteigne p
  n_kept = int((torch.cumsum(probabilities, dim=0) < p).sum()) + 1
  sampled = torch.multinomial(probabilities[:n_kept], num_samples=1)[0]
  return indices[0, sampled]


generations = test_decoding(k_p_sampling, 50, k=20, p=0.85)
print(f"3-grammes distincts : {statistics.mean(map(distinct_3grams, generations)):.2f}")

## Génération avec `generate`

En pratique, on n'écrit pas la boucle de décodage : la méthode `generate` des modèles de `transformers` implémente toutes ces stratégies, et d'autres, comme la recherche en faisceau (*beam search*). Elle prend les indices du prompt et un masque d'attention de même forme (1 pour un vrai token, 0 pour une position de remplissage : ici, que des 1), et renvoie les indices du prompt suivis des tokens générés. Ses paramètres règlent le choix du token suivant : `do_sample=False` pour le décodage glouton, `do_sample=True` pour l'échantillonnage, avec `temperature`, `top_k` et `top_p`.

*Codez la fonction `transformers_generate(prompt, length, **settings)`, qui génère `length` tokens après `prompt` avec `model.generate`, en lui passant les réglages `settings`, et renvoie le texte. Testez-la avec `test_decoding`, avec une centaine de tokens par prompt : d'abord avec un top-p sampling restreint aux 30 tokens les plus probables, de noyau `p = 0.95`, puis avec le décodage glouton. Comparez leur part de 3-grammes distincts.*

**Documentation utile**

- [`generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)
- [`GenerationConfig`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationConfig)

In [ ]:
def transformers_generate(prompt: str, length: int, **settings) -> str: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `token_ids = encode(prompt)`, et `torch.ones_like(token_ids)` pour le masque d'attention (`attention_mask`)
- `max_new_tokens=length` : le nombre de tokens à générer, sans compter le prompt
- `max_length=None` : la configuration de génération de BelGPT-2 limite le texte à 20 tokens, prompt compris ; on retire cette limite, pour que seul `max_new_tokens` compte (sinon, `transformers` avertit à chaque appel)
- `pad_token_id=start_token_id, eos_token_id=start_token_id` : les symboles de remplissage et de fin, que la configuration de BelGPT-2 donne faux (voir « Inspection du modèle »)
- `**settings` transmet les réglages : `{"do_sample": True, "top_k": 30, "top_p": 0.95}` pour le top-p sampling, `{"do_sample": False}` pour le décodage glouton
- `decode(generated)` : `generate` renvoie un tenseur de forme `(1, tokens du prompt + length)`, qui commence par le prompt

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def transformers_generate(prompt: str, length: int, **settings) -> str:
  token_ids = encode(prompt)
  generated = model.generate(
    token_ids,
    attention_mask=...,
    max_new_tokens=...,
    max_length=None,
    pad_token_id=start_token_id,
    eos_token_id=start_token_id,
    **settings,
  )
  return ...


for settings in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = test_decoding(transformers_generate, 100, **settings)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{settings} : {repetition:.2f} de 3-grammes distincts")
```

### Solution

In [ ]:
def transformers_generate(prompt: str, length: int, **settings) -> str:
  token_ids = encode(prompt)
  generated = model.generate(
    token_ids,
    attention_mask=torch.ones_like(token_ids),
    max_new_tokens=length,
    max_length=None,
    pad_token_id=start_token_id,
    eos_token_id=start_token_id,
    **settings,
  )
  return decode(generated)


for settings in [{"do_sample": True, "top_k": 30, "top_p": 0.95}, {"do_sample": False}]:
  generations = test_decoding(transformers_generate, 100, **settings)
  repetition = statistics.mean(map(distinct_3grams, generations))
  print(f"{settings} : {repetition:.2f} de 3-grammes distincts")

Sur une centaine de tokens, l'écart se creuse : la plupart des générations gloutonnes finissent par boucler, et leur part de 3-grammes distincts chute, alors que le top-p sampling reste varié. C'est pourquoi les modèles de langage sont presque toujours décodés par échantillonnage, avec une température, un top-k et un top-p réglés pour l'usage visé.